In [1]:
#1. Pathway Hierarchy test:
from src.builders.SQL.SQLCaller import SQL_ETL


UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'

sql_caller = SQL_ETL(run_id=UUID)

sql_caller.sql_state.ensure_staging_ready(table_name="PathwayData")

python-dotenv could not parse statement starting at line 2


True

In [1]:
import requests
from collections import defaultdict

pkg = "pathwayoracle"
data = requests.get(f"https://pypistats.org/api/packages/{pkg}/overall").json()["data"]

totals = defaultdict(int)
for row in data:
    totals[row["category"]] += row["downloads"]

mirrors = totals["with_mirrors"] - totals["without_mirrors"]
print(f"real: {totals['without_mirrors']:,}  mirrors: {mirrors:,}")

real: 277  mirrors: 2,838


In [ ]:
#1. Pathway Hierarchy test:
from src.builders.SQL.SQLCaller import SQL_ETL
from src.parsers.KEGGCaller import KEGG_ETL
from src.workflow.consumers import *
from src.workflow.producers import *

UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'
kegg_caller = KEGG_ETL()
sql_caller = SQL_ETL(run_id=UUID)
stream_data = produce_pathway_hierarchy(kegg_caller=kegg_caller)
staged_result = consume_pathway_hierarchy(data=stream_data, sql_caller=sql_caller)

In [6]:
from src.builders.SQL.SQLCaller import SQL_ETL
from src.parsers.KEGGCaller import KEGG_ETL
from src.workflow.consumers import *
from src.workflow.producers import *

UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'
sql_caller = SQL_ETL(run_id=UUID)

manager = sql_caller.sql_state.get_manager('kegg_class')

for sql in manager.strategy.upsert_data(manager.schema):
    print(sql)

NotImplementedError: kegg_class uses direct hierarchical insertion -- call SQL_ETL.insert_hierarchical(), not stage_data()/upsert_data().

In [ ]:
from src.builders.SQL.SQLCaller import SQL_ETL
from src.parsers.KEGGCaller import KEGG_ETL
from src.workflow.consumers import *
from src.workflow.producers import *

UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'
sql_caller = SQL_ETL(run_id=UUID)

sql_caller.insert_hierarchical


In [12]:
from src.parsers.KEGGCaller import KEGG_ETL

kegg_caller = KEGG_ETL()
kgml_file = kegg_caller.kegg_state.fetch_pathway_kgml('hsa00010')

kegg_caller.kegg_state.download_kgml_temp_file('hsa00010', kgml_file)

In [5]:
from src.parsers.KEGGCaller import KEGG_ETL


kegg_caller = KEGG_ETL()

for record in kegg_caller.parse_kgml_structure('hsa00010'):
    sorted_source_relation = sorted(record.entity_path_mem, key=lambda r: r.entity_id)
    for interaction in sorted_source_relation:
        print(interaction)
    break

EntityPathMem(entity_id='10327', pathway_id='hsa00010')
EntityPathMem(entity_id='124', pathway_id='hsa00010')
EntityPathMem(entity_id='125', pathway_id='hsa00010')
EntityPathMem(entity_id='126', pathway_id='hsa00010')
EntityPathMem(entity_id='127', pathway_id='hsa00010')
EntityPathMem(entity_id='128', pathway_id='hsa00010')
EntityPathMem(entity_id='130', pathway_id='hsa00010')
EntityPathMem(entity_id='130589', pathway_id='hsa00010')
EntityPathMem(entity_id='131', pathway_id='hsa00010')
EntityPathMem(entity_id='160287', pathway_id='hsa00010')
EntityPathMem(entity_id='1737', pathway_id='hsa00010')
EntityPathMem(entity_id='1738', pathway_id='hsa00010')
EntityPathMem(entity_id='2023', pathway_id='hsa00010')
EntityPathMem(entity_id='2026', pathway_id='hsa00010')
EntityPathMem(entity_id='2027', pathway_id='hsa00010')
EntityPathMem(entity_id='217', pathway_id='hsa00010')
EntityPathMem(entity_id='218', pathway_id='hsa00010')
EntityPathMem(entity_id='219', pathway_id='hsa00010')
EntityPathMem(e

In [1]:
from pprint import pprint
from src.builders.SQL.schema.table_registry import table_managers
from src.builders.SQL.schema.definitions import table_schemas
from src.builders.SQL.schema.table_utils import resolve_staging_plan


for table_name, schema in table_schemas.items():
    manager = table_managers[table_name]
    staging_create_sql = manager.strategy._build_staging_create(schema, f'staging.{table_name}')

    print(staging_create_sql)


        IF NOT EXISTS (
            SELECT 1 FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'PathwayIds'
        )
        BEGIN
            CREATE TABLE staging.PathwayIds (
   
)
        END
        

        IF NOT EXISTS (
            SELECT 1 FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'PathwayKGMLMeta'
        )
        BEGIN
            CREATE TABLE staging.PathwayKGMLMeta (
   
)
        END
        

        IF NOT EXISTS (
            SELECT 1 FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'entities'
        )
        BEGIN
            CREATE TABLE staging.entities (
   
)
        END
        

        IF NOT EXISTS (
            SELECT 1 FROM INFORMATION_SCHEMA.TABLES
            WHERE TABLE_SCHEMA = 'staging' AND TABLE_NAME = 'EntityPathMem'
        )
        BEGIN
            CREATE TABLE staging.EntityPathMem (
   identity_hash VARBINARY(

In [10]:
for table_name, schema in table_schemas.items():
    if table_name != "EntityPathMem":
        continue
    manager = table_managers[table_name]
    pprint(manager.strategy.stage_data(manager.schema))
    

StagingQuery(sql='\n'
                 '        INSERT INTO staging.EntityPathMem (pathway_id, '
                 'entity_id)\n'
                 '        SELECT ?, ?\n'
                 '        WHERE NOT EXISTS (\n'
                 '            SELECT 1 FROM staging.EntityPathMem t\n'
                 "            WHERE ISNULL(t.pathway_id, N'') = ISNULL(?, N'') "
                 "AND ISNULL(t.entity_id, N'') = ISNULL(?, N'')\n"
                 '        )\n'
                 '        ',
             insert_cols=('pathway_id', 'entity_id'),
             match_cols=('pathway_id', 'entity_id'),
             num_params=4)


In [2]:
resolve_staging_plan(table_schemas['OrthoData'])

[ColumnPlan(production_col='ortho_id', staging_col='ortho_id', staging_type='VARCHAR(20)', deffered_fk=None, reused_column=False),
 ColumnPlan(production_col='ortho_name', staging_col='ortho_name', staging_type='VARCHAR(20)', deffered_fk=None, reused_column=False),
 ColumnPlan(production_col='full_name', staging_col='full_name', staging_type='TEXT', deffered_fk=None, reused_column=False)]

In [ ]:
@sql_safe(default=None, label="staging data into table")
def stage_data(self, target_table: str, data: Sequence[BaseSQLObject]):
    if not data:
        logger.error(f"No {target_table} data for this batch")
        return

    self.sql_state.ensure_staging_ready(target_table)
    manager = self.sql_state.get_manager(target_table)
    query = manager.strategy.stage_data(manager.schema)

    rows = [asdict(row) for row in data]
    try:
        params = [
            tuple(row[c] for c in query.insert_cols) + tuple(row[c] for c in query.match_cols)
            for row in rows
        ]
    except KeyError as e:
        raise ValueError(
            f"{target_table}: row is missing field {e} required by staging query "
            f"(insert_cols={query.insert_cols}, match_cols={query.match_cols}) -- "
            f"check that the BaseSQLObject model's field names match the staging "
            f"column names produced by resolve_staging_plan."
        )

    if params and len(params[0]) != query.num_params:
        raise ValueError(
            f"Param count mismatch for {target_table}: "
            f"query expects {query.num_params}, got {len(params[0])}"
        )

    with self.conn.cursor() as cursor:
        cursor.executemany(query.sql, params)
    self.conn.commit()
    logger.info(f"Staged {len(data)} rows into staging.{target_table} (duplicates skipped).")
    return True

In [6]:
from src.parsers.KEGGCaller import KEGG_ETL
from bs4 import BeautifulSoup
from typing import Iterator
from src.models.kegg import *

def _parse_kgml_to_entry_map(xml_content)-> Iterator[PathwayKGMLRecord]:
    soup = BeautifulSoup(xml_content, "xml")

    pathway_entry = soup.find("pathway")
    pathway_id = pathway_entry.get("name").split("path:")[1]
    pathway = Pathway(pathway_id= pathway_id,
                      description= pathway_entry.get("title"))


    entry_map : dict[str, dict[str, list[Entity]]] = {}
    for entry in soup.find_all("entry"):
        entry_id = entry.get("id")
        entity_ids = entry.get("name").split()
        entity_type = entry.get("type")

        if entity_type == "map":
            continue

        entry_map[entry_id] = {
            "entity_path_mem": [EntityPathMem(entity_id, pathway_id) for entity_id in entity_ids],
        }
        if entity_type == "gene":
            entry_map[entry_id]["entities"] = [Entity(entity_id.split(":")[1], EntityType.GENE) for entity_id in entity_ids]
        elif entity_type == "ortholog":
            entry_map[entry_id]["entities"] = [Entity(entity_id.split(":")[1], EntityType.ORTHOLOG) for entity_id in entity_ids]
        elif entity_type == "compound":
            entry_map[entry_id]["entities"] = [Entity(entity_id.split(":")[1], EntityType.COMPOUND) for entity_id in entity_ids]
        else:
            raise ValueError("Unexpected entry type")

    interactions = []
    for rel in soup.find_all("relation"):
        source = rel.get("entry1")
        target = rel.get("entry2")
        relation_type = rel.find("subtype").get("name")
        if source in entry_map and target in entry_map:
            source_synonyms = entry_map[source]["entities"]
            target_synonyms = entry_map[target]["entities"]
            for source_syn in source_synonyms:
                for target_syn in target_synonyms:
                    interactions.append(Interaction(source_id=source_syn.entity_id,
                                                    target_id=target_syn.entity_id,
                                                    relation_type=relation_type,
                                                    pathway_id= pathway_id))

    reaction_participants = []
    for rx in soup.find_all("reaction"):
        rx_entry_id = rx.get("id")
        reaction_id = rx.get("name").strip("rn:")
        if rx_entry_id in entry_map:
            #For catalysts
            for catalyst in entry_map[rx_entry_id]["entities"]:
                reaction_participants.append(ReactionP(
                    reaction_id= reaction_id,
                    entity_id = catalyst.entity_id,
                    role = "catalyst",
                    pathway_id= pathway_id
                ))
            substrates = rx.find_all("substrate")
            products = rx.find_all("products")
            for substrate in substrates:
                reaction_participants.append(ReactionP(
                    reaction_id=reaction_id,
                    entity_id=substrate.get("name").strip("cpd:"),
                    role = "substrate",
                    pathway_id = pathway_id
                ))
            for product in products:
                reaction_participants.append(ReactionP(
                    reaction_id=reaction_id,
                    entity_id=product.get("name").strip("cpd:"),
                    role="product",
                    pathway_id=pathway_id
                ))

    entities = [entity for entity_id, entry_dicts in entry_map.items() for entity in entry_dicts["entities"]]
    entity_path_mem = [entity_path for entity_id, entry_dicts in entry_map.items() for entity_path in entry_dicts["entity_path_mem"]]
    return PathwayKGMLRecord(pathway, entities, entity_path_mem, interactions, reaction_participants)

kegg_caller = KEGG_ETL()
xml_content = kegg_caller.kegg_state.read_kgml_temp_file("hsa00010")
record = _parse_kgml_to_entry_map(xml_content)

In [7]:
from pprint import pprint


pprint(record)

PathwayKGMLRecord(pathway=Pathway(pathway_id='hsa00010',
                                  description='Glycolysis / Gluconeogenesis'),
                  entities=[Entity(entity_id='226',
                                   entity_type=<EntityType.GENE: 'gene'>),
                            Entity(entity_id='229',
                                   entity_type=<EntityType.GENE: 'gene'>),
                            Entity(entity_id='230',
                                   entity_type=<EntityType.GENE: 'gene'>),
                            Entity(entity_id='217',
                                   entity_type=<EntityType.GENE: 'gene'>),
                            Entity(entity_id='219',
                                   entity_type=<EntityType.GENE: 'gene'>),
                            Entity(entity_id='223',
                                   entity_type=<EntityType.GENE: 'gene'>),
                            Entity(entity_id='224',
                                   entity_type=<En

In [5]:
from src.workflow.producers import *
from src.workflow.consumers import *
from src.builders.SQL.SQLCaller import SQL_ETL
from src.parsers.KEGGCaller import KEGG_ETL

UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'
sql_caller = SQL_ETL(run_id=UUID)
sql_caller._build_merge_query(target_table='kegg_class')

'\n        MERGE INTO dbo.kegg_class AS target\n        USING staging.kegg_class AS source\n        ON target.class_id = source.class_id\n        WHEN MATCHED THEN UPDATE SET target.name = source.name, target.parent_id = source.parent_id\n        WHEN NOT MATCHED THEN INSERT (class_id, name, parent_id) VALUES (source.class_id, source.name, source.parent_id)\n        ;\n        '

In [1]:
from src.builders.SQL.schema import resolve_clear_order, table_schemas, resolve_staging_plan
from src.builders.SQL.SQLCaller import SQL_ETL

UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'

sql_caller = SQL_ETL(run_id=UUID)
sql_caller._build_staging_query(target_table='kegg_class')

python-dotenv could not parse statement starting at line 2


("\n        INSERT INTO staging.kegg_class (name, parent_name)\n        SELECT ?, ?\n        WHERE NOT EXISTS (\n            SELECT 1 FROM staging.kegg_class t\n            WHERE ISNULL(t.name, N'') = ISNULL(?, N'') AND ISNULL(t.parent_name, N'') = ISNULL(?, N'')\n        )\n        ",
 4)

In [2]:
from src.workflow.producers import *
from src.workflow.consumers import *
from src.builders.SQL.SQLCaller import SQL_ETL
from src.parsers.KEGGCaller import KEGG_ETL

UUID = '5B146EBC-98C5-4242-A049-A5331A641CB4'

kegg_caller = KEGG_ETL()
sql_caller = SQL_ETL(run_id=UUID)
stream_data = produce_pathway_ids(kegg_caller=kegg_caller)
staged_result = consume_pathway_ids(data=stream_data, sql_caller=sql_caller)


        INSERT INTO staging.PathwayIds (pathway_id, name)
        SELECT ?, ?
        WHERE NOT EXISTS (
            SELECT 1 FROM staging.PathwayIds t
            WHERE t.pathway_id = ?
        )
         [('hsa01100', 'Metabolic pathways - Homo sapiens (human)', 'hsa01100'), ('hsa01200', 'Carbon metabolism - Homo sapiens (human)', 'hsa01200'), ('hsa01210', '2-Oxocarboxylic acid metabolism - Homo sapiens (human)', 'hsa01210'), ('hsa01212', 'Fatty acid metabolism - Homo sapiens (human)', 'hsa01212'), ('hsa01230', 'Biosynthesis of amino acids - Homo sapiens (human)', 'hsa01230'), ('hsa01232', 'Nucleotide metabolism - Homo sapiens (human)', 'hsa01232'), ('hsa01250', 'Biosynthesis of nucleotide sugars - Homo sapiens (human)', 'hsa01250'), ('hsa01240', 'Biosynthesis of cofactors - Homo sapiens (human)', 'hsa01240'), ('hsa01320', 'Sulfur cycle - Homo sapiens (human)', 'hsa01320'), ('hsa00010', 'Glycolysis / Gluconeogenesis - Homo sapiens (human)', 'hsa00010'), ('hsa00020', 'Citrate cycle (

In [2]:
all_nodes = kegg_caller.fetch_brite_hierarchy()
kegg_caller.flatten_brite(all_nodes)

[('br08901', None),
 ('Metabolism', 'br08901'),
 ('Global and overview maps', 'Metabolism'),
 ('Metabolic pathways', 'Global and overview maps'),
 ('Biosynthesis of secondary metabolites', 'Global and overview maps'),
 ('Microbial metabolism in diverse environments', 'Global and overview maps'),
 ('Carbon metabolism', 'Global and overview maps'),
 ('2-Oxocarboxylic acid metabolism', 'Global and overview maps'),
 ('Fatty acid metabolism', 'Global and overview maps'),
 ('Biosynthesis of amino acids', 'Global and overview maps'),
 ('Nucleotide metabolism', 'Global and overview maps'),
 ('Biosynthesis of nucleotide sugars', 'Global and overview maps'),
 ('Biosynthesis of cofactors', 'Global and overview maps'),
 ('Degradation of aromatic compounds', 'Global and overview maps'),
 ('Nitrogen cycle', 'Global and overview maps'),
 ('Sulfur cycle', 'Global and overview maps'),
 ('Carbohydrate metabolism', 'Metabolism'),
 ('Glycolysis / Gluconeogenesis', 'Carbohydrate metabolism'),
 ('Citrate cy

In [1]:
from src.builders.Neo4j.schema.nodes import Ontology, Gene


TERM_LABEL, TERM_KEY = Ontology.__label__, Ontology.__key__      # "Ontology", "id"
TERM_NAME_PROP = "name"
TERM_NAMESPACE_PROP = "hasOBONamespace"                          # see note above

GENE_LABEL, GENE_KEY = Gene.__label__, Gene.__key__               # "Gene", "id"

# True path rule applies to these hierarchy relations only. Real predicate
# names confirmed against go-basic.json's edges (is_a, and BFO_0000050 /
# RO_0002211-3 which OntoStateManager.sync_ontology_structure remaps to
# part_of / regulates / negatively_regulates / positively_regulates).
HIERARCHY_RELS = ("is_a", "part_of")
SIBLING_REL = "is_a"   # siblings = terms sharing an is_a parent

Q_ANNOTATIONS = f"""
MATCH (g:{GENE_LABEL})-[a]->(t:{TERM_LABEL})
RETURN g.{GENE_KEY} AS gene, t.{TERM_KEY} AS term,
       type(a) AS qualifier, a.evidence AS evidence
"""

In [4]:
from neo4j import RoutingControl
from src.builders.Neo4j.Neo4jCaller import Neo4j_ETL
from src.builders.SQL.SQLCaller import SQL_ETL
from src.builders.Qdrant.dataset import _load_protein_data
import os
from dotenv import load_dotenv

load_dotenv()

UUID = os.getenv('uuid')

neo4j_caller = Neo4j_ETL()
sql_caller = SQL_ETL(run_id=UUID)

def run(q: str) -> list[dict]:
    records, _, _ = neo4j_caller.driver.execute_query(q, database_=None, routing_=RoutingControl.READ)
    return [r.data() for r in records]

entrez_to_uniprots, _ = _load_protein_data(sql_caller)
annotated = {a["gene"] for a in run(Q_ANNOTATIONS)}
print(len(annotated - set(entrez_to_uniprots)), "annotated genes with no usable mapping")

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 2


2793 annotated genes with no usable mapping


In [5]:
annotated = {a["gene"] for a in run(Q_ANNOTATIONS)}
print(len(annotated), "annotated genes total;", 
      len(annotated - set(entrez_to_uniprots)), "dropped for lack of function text")


8497 annotated genes total; 2793 dropped for lack of function text


In [7]:
from dataclasses import dataclass, field

@dataclass
class Config:
    # Experimental, high-throughput, and curator-statement codes. Add "IBA"
    # (phylogenetic) for much more coverage at slightly lower precision.
    keep_evidence: frozenset = frozenset({
        "EXP", "IDA", "IPI", "IMP", "IGI", "IEP",
        "HTP", "HDA", "HMP", "HGI", "HEP", "TAS", "IC"})
    # Qualifiers too indirect to mean "this text describes this term"
    drop_qualifier_prefixes: tuple = ("acts_upstream_of", "colocalizes_with")
    # UniProt function text mostly describes MF and BP, rarely location (CC)
    keep_namespaces: frozenset = frozenset({"molecular_function", "biological_process"})
    blocklist: frozenset = frozenset({"GO_0005515"})  # "protein binding": uninformative
    min_pos: int = 5        # too few genes -> too few training pairs
    max_pos: int = 300      # too many -> term is too generic to be a useful query
    pos_per_anchor: int = 30
    p_curated_neg: float = 0.5  # how often to prefer a curated NOT when one exists
    test_frac: float = 0.1
    seed: int = 13

In [8]:
KEEP = Config().keep_evidence
strong = set()
for a in run(Q_ANNOTATIONS):
    ev = a["evidence"]
    evs = set(ev) if isinstance(ev, (list, tuple)) else {ev}
    if evs & KEEP:
        strong.add(a["gene"])
lost = strong - set(entrez_to_uniprots)
print(len(strong), "genes with keep-evidence annotations;",
      len(lost), f"({100*len(lost)/len(strong):.0f}%) have no function text")


8024 genes with keep-evidence annotations; 2686 (33%) have no function text


In [2]:
from neo4j import RoutingControl
from src.builders.Neo4j.Neo4jCaller import Neo4j_ETL
from src.builders.SQL.SQLCaller import SQL_ETL
from src.builders.Qdrant.dataset import _load_protein_data
import os
from dotenv import load_dotenv

load_dotenv()

UUID = os.getenv('uuid')

neo4j_caller = Neo4j_ETL()
sql_caller = SQL_ETL(run_id=UUID)

python-dotenv could not parse statement starting at line 2
python-dotenv could not parse statement starting at line 2


In [3]:
from src.builders.Qdrant.dataset import build_dataset, extract


extracted = extract(neo4j_caller, sql_caller)      # KB machine only
stats = build_dataset(**extracted).stats

hist = stats["pos_size_near_min"]
for m in (3, 4, 5):
    print(m, stats["anchors_total"] + sum(hist[n] for n in range(m, 5)))



3 6666
4 5531
5 4769
